# Gen grids from vids folder (After running gen_vids.py)

In [11]:
import subprocess, os, glob, tqdm
ax = 1
# videos = glob.glob("./vids/all_outputs/res/*axis=2_combined.mp4")
videos = glob.glob(f"./vids_rot1/all_outputs/res/*axis={ax}.mp4")

# Grid configuration
grid_h = 4  # Number of rows in the grid
grid_w = 9  # Number of columns in the grid
n_grid = 4

# Ensure the number of videos matches the grid size
if len(videos) < grid_h * grid_w:
    raise ValueError("Not enough videos to fill the grid.")

# Use pre-defined samples
# use_predef = True
# Pair-id related to id in "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/selected_rotate_RT_60065.json"
# videos_to_use_predef = [
#     [2, 39, 56, 80, 101, 111, 127, 130, 132, 148, 182, 183, 198, 219, 235, 240, 276, 308, 348, 364, 367, 373, 415, 443, 454]
# ]



exclude = ['67690.jpg', '66995.jpg', '66138.jpg', '62747.jpg', '60306.jpg', '64610.jpg', '60400.jpg', 
           '62553.jpg', '66130.jpg', '66426.jpg', '64360.jpg', '60799.jpg', '61297.jpg', '66117.jpg', 
           '68537.jpg', '63059.jpg', '64375.jpg', '69348.jpg', '66128.jpg', '67887.jpg', '66804.jpg',
        ]
print(len(videos), len(exclude))
for e in exclude:
    videos = [v for v in videos if e not in v]
print(len(videos))
# Shuffle the videos list order
import random
random.seed(4725091995)
# print(videos)
random.shuffle(videos)
# print(videos)

for i in range(n_grid):
    os.makedirs(f"./vids_rot1/all_outputs/grids_{ax}/", exist_ok=True)
    # Create temporary intermediate files for each row
    intermediate_files = []
    # if use_predef:
    #     videos_to_use = videos_to_use_predef[i]
    # else:
    videos_to_use = videos[i*grid_h*grid_w:(i+1)*grid_h*grid_w]

    for row in tqdm.tqdm(range(grid_h)):
        # Get the videos for this row
        row_videos = videos_to_use[row * grid_w:(row + 1) * grid_w]

        # Build the ffmpeg hstack command for this row
        input_cmds = []
        for idx, video in enumerate(row_videos):
            input_cmds += ["-i", video]

        hstack_filter =  f'-filter_complex "hstack=inputs={len(row_videos)}"'
        output_row = f"./vids_rot1/all_outputs/grids_{ax}/row_{row}.mp4"
        try:
            subprocess.run(f"ffmpeg {' '.join(input_cmds)} {hstack_filter} -y {output_row}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError as e:
            print(e)
            exit(1)
            
        intermediate_files.append(output_row)

    # Build the ffmpeg vstack command
    input_cmds = []
    for row_file in intermediate_files:
        input_cmds += ["-i", row_file]
    vstack_filter = f'-filter_complex "vstack=inputs={len(intermediate_files)}"'
    output_grid = f"./vids_rot1/all_outputs/grids_{ax}/grid{i}.mp4"
    try:
        subprocess.run(f"ffmpeg {' '.join(input_cmds)} {vstack_filter} -y {output_grid}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print(e)
        exit(1)

214 21
193


100%|██████████| 4/4 [00:16<00:00,  4.14s/it]


# Get the render ball

In [7]:
# Compute the timestamp for setting ppt
fps = 24
s1_relight_start = 59 + 20
s2_reshadow_diffuse = 30   # forward and reverse
s3_reshadow_tomax = 30 + 30
s4_relight_at_max = 59 + 59 # forward and reverse
s5_reshadow_to_start = 30
s6_relight_at_start = 59

# Compute the timestamp for setting ppt
elapsed_time = 0
for step in [s1_relight_start, s2_reshadow_diffuse, s3_reshadow_tomax, s4_relight_at_max, s5_reshadow_to_start, s6_relight_at_start]:
    elapsed_time += step/fps
    print("[#] Elapsed time: ", elapsed_time)



[#] Elapsed time:  3.2916666666666665
[#] Elapsed time:  4.541666666666666
[#] Elapsed time:  7.041666666666666
[#] Elapsed time:  11.958333333333332
[#] Elapsed time:  13.208333333333332
[#] Elapsed time:  15.666666666666666
